# Calibrated vs Non-Calibrated TabPFN — OOF Prediction Analysis

Runs 5-fold CV twice on the `gbm` feature set:
- **Calibrated**: TabPFN → Tweedie GLM recalibration layer (Option B nested OOF)
- **Raw**: TabPFN direct output, no recalibration

Key observation from prior runs:
```
Calibrated:     tweedie_dev=90.2,  gini=0.2844, rmse=14498, mae=411
Non-calibrated: tweedie_dev=64M!!, gini=0.2844, rmse=14498, mae=171
```
Identical Gini and RMSE → the *ranking* is exactly the same.  
Catastrophic Tweedie deviance → the *scale/distribution* is completely wrong.

In [ ]:
import sys, os
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from src.data.load_insurance import load_processed, get_dev_holdout
from src.features.pipelines import GBMFeaturePipeline
from src.models.tabpfn_wrapper import TabPFNWrapper
from src.evaluation.cv_engine import run_cv
from src.metrics import tweedie_deviance

df = load_processed()
splits = get_dev_holdout(df)
X_dev, X_holdout   = splits['X_dev'],   splits['X_holdout']
y_dev, y_holdout   = splits['y_dev'],   splits['y_holdout']
exp_dev, exp_holdout = splits['exp_dev'], splits['exp_holdout']
cv_folds           = splits['cv_folds']

print(f'Dev: {len(X_dev):,}  Holdout: {len(X_holdout):,}')

In [ ]:
# Run CV twice — calibrated and raw
# n_train_max=10_000, gbm features, tweedie approach

common_kwargs = dict(
    feature_pipeline_factory=GBMFeaturePipeline,
    X_dev=X_dev, y_dev=y_dev, exposure_dev=exp_dev,
    cv_folds=cv_folds,
    approach='tweedie',
    features_label='gbm',
    save=False,
)

print('--- Running CALIBRATED (recalibrate=True) ---')
res_cal = run_cv(
    model_factory=lambda: TabPFNWrapper(task='regression', n_train_max=10_000,
                                        exposure_strategy='feature', predict_batch_size=5_000),
    model_name='TabPFN_10K_calibrated',
    tabpfn_recalibrate=True,
    **common_kwargs,
)

print('\n--- Running RAW (recalibrate=False) ---')
res_raw = run_cv(
    model_factory=lambda: TabPFNWrapper(task='regression', n_train_max=10_000,
                                        exposure_strategy='feature', predict_batch_size=5_000),
    model_name='TabPFN_10K_raw',
    tabpfn_recalibrate=False,
    **common_kwargs,
)

preds_cal = res_cal['oof_predictions']
preds_raw = res_raw['oof_predictions']
y_true    = y_dev['pure_premium'].values

print('\n=== Mean CV metrics ===')
print('Calibrated:', res_cal['mean_metrics'])
print('Raw:       ', res_raw['mean_metrics'])

In [ ]:

# ── Visual analysis ────────────────────────────────────────────────────────
nonzero = y_true > 0
y_nz   = y_true[nonzero]
cal_nz = preds_cal[nonzero]
raw_nz = preds_raw[nonzero]

fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(2, 2, hspace=0.42, wspace=0.35)

# ── A: Prediction distributions (log x-axis, counts) ──────────────────────
# Bins must span ALL three series — raw TabPFN may predict on a very different
# scale. density=True with log-spaced bins is visually misleading (varying bin
# widths inflate/deflate bars), so we use counts and yscale('log') instead.
ax_a = fig.add_subplot(gs[0, 0])
all_vals = np.concatenate([y_nz, cal_nz, raw_nz])
lo = np.log10(max(all_vals[all_vals > 0].min(), 1e-2))
hi = np.log10(all_vals.max())
bins = np.logspace(lo, hi, 60)
ax_a.hist(y_nz,   bins=bins, alpha=0.5, label='Actual',     color='steelblue')
ax_a.hist(cal_nz, bins=bins, alpha=0.6, label='Calibrated', color='seagreen')
ax_a.hist(raw_nz, bins=bins, alpha=0.5, label='Raw TabPFN', color='tomato')
ax_a.set_xscale('log')
ax_a.set_yscale('log')
ax_a.set_xlabel('Pure Premium (log scale)')
ax_a.set_ylabel('Count (log)')
ax_a.set_title('A — Prediction Distributions (claims only)')
ax_a.legend()

# ── B: Predicted vs actual (log-log) ───────────────────────────────────────
# Reference line must cover the range of ALL predictions, not just actuals.
ax_b = fig.add_subplot(gs[0, 1])
rng = np.random.default_rng(0)
sample = rng.choice(len(y_nz), size=min(3000, len(y_nz)), replace=False)
ax_b.scatter(y_nz[sample], raw_nz[sample], alpha=0.25, s=8, color='tomato',   label='Raw')
ax_b.scatter(y_nz[sample], cal_nz[sample], alpha=0.25, s=8, color='seagreen', label='Calibrated')
all_pos = np.concatenate([y_nz, cal_nz, raw_nz])
all_pos = all_pos[all_pos > 0]
ref_lo  = all_pos.min()
ref_hi  = all_pos.max()
ax_b.plot([ref_lo, ref_hi], [ref_lo, ref_hi], 'k--', lw=1, label='Perfect')
ax_b.set_xscale('log')
ax_b.set_yscale('log')
ax_b.set_xlabel('Actual Pure Premium')
ax_b.set_ylabel('Predicted Pure Premium')
ax_b.set_title('B — Predicted vs Actual (log-log, 3K sample)')
ax_b.legend(markerscale=3)

# ── C: Rank lift plot ───────────────────────────────────────────────────────
ax_c = fig.add_subplot(gs[1, 0])
n_bins = 50
for preds, label, color in [(preds_cal, 'Calibrated', 'seagreen'), (preds_raw, 'Raw TabPFN', 'tomato')]:
    order   = np.argsort(preds)
    chunks  = np.array_split(order, n_bins)
    mean_pp = [y_true[c].mean() for c in chunks]
    ax_c.plot(range(n_bins), mean_pp, marker='o', ms=3, label=label, color=color)
ax_c.set_xlabel('Bin (sorted by model score, low → high risk)')
ax_c.set_ylabel('Mean actual pure premium in bin')
ax_c.set_title('C — Ranking quality (overlap = identical Gini)')
ax_c.legend()

# ── D: Relative error — log-y to show the -1 spike for raw model ──────────
# Raw TabPFN likely piles up near rel_err = -1 (predicts ~0, actual is large).
# clip left at -1.5 to keep the spike visible; log-y reveals both models.
ax_d = fig.add_subplot(gs[1, 1])
rel_err_cal = (cal_nz - y_nz) / y_nz
rel_err_raw = (raw_nz - y_nz) / y_nz
bins_err = np.linspace(-1.5, 5, 100)
ax_d.hist(np.clip(rel_err_cal, -1.5, 5), bins=bins_err, alpha=0.6,
          label=f'Calibrated  median={np.median(rel_err_cal):.2f}', color='seagreen')
ax_d.hist(np.clip(rel_err_raw, -1.5, 5), bins=bins_err, alpha=0.5,
          label=f'Raw TabPFN  median={np.median(rel_err_raw):.2f}', color='tomato')
ax_d.axvline(0, color='k', lw=1, ls='--', label='zero error')
ax_d.set_yscale('log')
ax_d.set_xlabel('Relative error  (pred − actual) / actual')
ax_d.set_ylabel('Count (log)')
ax_d.set_title('D — Relative error (claims only, log-y)')
ax_d.legend()

plt.suptitle('Calibrated vs Raw TabPFN OOF Predictions — GBM features, 10K', fontsize=14, y=1.01)
plt.savefig('../results/recalibration_analysis.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved → results/recalibration_analysis.png')


In [ ]:
# ── Summary table ──────────────────────────────────────────────────────────
from src.metrics import gini_coefficient, rmse, mae, poisson_deviance

rows = []
for label, preds in [('Calibrated', preds_cal), ('Raw TabPFN', preds_raw)]:
    rows.append({
        'Model': label,
        'Tweedie dev (1.5)': round(tweedie_deviance(y_true, preds, power=1.5, sample_weight=exp_dev.values), 2),
        'Gini': round(gini_coefficient(y_true, preds), 4),
        'RMSE': round(rmse(y_true, preds, sample_weight=exp_dev.values), 2),
        'MAE': round(mae(y_true, preds, sample_weight=exp_dev.values), 2),
        'Median pred (claims)': round(np.median(preds[nonzero]), 2),
        'Median actual (claims)': round(np.median(y_true[nonzero]), 2),
    })

pd.DataFrame(rows).set_index('Model')